## 自定义中间件

#### 加载环境

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

#### wrap_model_call

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware
from typing import Callable
from rich.pretty import pprint


@wrap_model_call
def measure_model_time(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:

    start = perf_counter()
    print("准备调用模型")

    try:
        response = handler(request)
        pprint(response.result[0])
        return response
    finally:
        elapsed = perf_counter() - start
        print(f"模型调用结束，耗时：{elapsed:.2f} 秒")


class WrapModelCallMiddleWare(AgentMiddleware):

    def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:

            start = perf_counter()
            print("准备调用模型")

            try:
                response = handler(request)
                pprint(response.result[0])
                return response
            finally:
                elapsed = perf_counter() - start
                print(f"模型调用结束，耗时：{elapsed:.2f} 秒")

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model='deepseek:deepseek-flash',
    middleware=[
        # measure_model_time,
        WrapModelCallMiddleWare(),
    ]
)

resp = agent.invoke({'messages': HumanMessage("你好")})

#### wrap_tool_call

In [ ]:
from langgraph.types import Command
from langchain.agents.middleware import wrap_tool_call, ModelRequest, ModelResponse, AgentMiddleware, ToolCallRequest
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from time import perf_counter

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city}天气晴天~"

@wrap_tool_call
def measure_tool_time(
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage]
) -> ToolMessage:

    start = perf_counter()
    print("准备调用工具")

    try:
        response = handler(request)
        print(response)
        return response
    finally:
        elapsed = perf_counter() - start
        print(f"工具调用结束，耗时：{elapsed:.2f} 秒")


class WrapToolCallMiddleWare(AgentMiddleware):
    def wrap_tool_call(
            self,
            request: ToolCallRequest,
            handler: Callable[[ToolCallRequest], ToolMessage | Command]
    ) -> ToolMessage | Command:

        start = perf_counter()
        print("准备调用工具")

        try:
            response = handler(request)
            print(response)
            return response
        finally:
            elapsed = perf_counter() - start
            print(f"工具调用结束，耗时：{elapsed:.2f} 秒")


In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[get_weather],
    middleware=[
        # measure_tool_time,
        WrapToolCallMiddleWare(),
    ]
)

resp = agent.invoke({'messages': HumanMessage("今天北京的天气如何")})
for msg in resp["messages"]:
    msg.pretty_print()